# Notebook 10: Multi-Class Classification Metrics

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 7 — Model Evaluation, Validation & Metrics**

---

## Why This Matters

Binary classification is straightforward: did the model say spam or not-spam? But what when you have **four categories** — spam, promotional, personal, work? Now you need to ask: is the model equally good at all four? Does it sacrifice performance on rare classes to boost the majority? Standard accuracy hides all of this.

Multi-class metrics let you **decompose** model performance class by class, and then **recombine** those class-level scores using different averaging strategies — each answering a different business question.

---

## Real-World Analogy: A Post Office With Four Sorting Bins

Imagine a post office with four mail bins: **Spam**, **Promotional**, **Personal**, **Work**. A sorting robot classifies every incoming letter.

- **Per-class metrics:** How good is the robot at each specific bin? Maybe it's great at Work mail but terrible at Personal.
- **Macro averaging:** Score the robot on each bin equally — even if 90% of mail is Work, the Personal bin still counts 25% of the score.
- **Micro averaging:** Score the robot on raw counts — since Work dominates, performance on Work mail dominates the score too.
- **Weighted averaging:** Score the robot proportionally — Work mail (most common) gets more weight; Personal (rarest) gets less.
- **Cohen's Kappa:** How much better is the robot than a random sorter who just guesses based on bin frequencies?

---

## Theme: Email Classification

Four classes: **Spam** (40%), **Promotional** (30%), **Personal** (20%), **Work** (10%)

This intentional imbalance lets us see macro, micro, and weighted metrics disagree — which is the most educational scenario.

## Section 1: Setup and Dataset Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    accuracy_score,
)
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11
sns.set_theme(style='whitegrid', palette='muted')

# Class definitions
CLASS_NAMES  = ['Spam', 'Promotional', 'Personal', 'Work']
CLASS_COLORS = ['tomato', 'steelblue', 'mediumseagreen', 'darkorchid']
N_CLASSES    = len(CLASS_NAMES)

print("Libraries loaded.")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# ---------------------------------------------------------------
# Generate 2,000 email samples with 4 imbalanced classes
# Class distribution: Spam 40%, Promotional 30%, Personal 20%, Work 10%
# ---------------------------------------------------------------
n_samples = 2000
class_probs = [0.40, 0.30, 0.20, 0.10]
class_counts = [int(n_samples * p) for p in class_probs]
# Fix rounding: ensure total = n_samples
class_counts[-1] += n_samples - sum(class_counts)

print("Class distribution:")
for name, count, prob in zip(CLASS_NAMES, class_counts, class_probs):
    print(f"  {name:15s}: {count:4d} samples ({prob*100:.0f}%)")

# Create class-conditional features (mean vectors differ by class)
# Features: word_count, link_count, sender_known, exclamation_marks, reply_count
feature_means = {
    0: [50,  8, 0.1, 5, 0.1],   # Spam: many links, many !, unknown sender
    1: [80,  4, 0.4, 2, 0.2],   # Promotional: moderate links, known sender
    2: [120, 1, 0.9, 1, 2.5],   # Personal: long, few links, known sender, replies
    3: [200, 2, 0.8, 0, 1.5],   # Work: longest, few links, very professional
}
feature_stds = [40, 3, 0.25, 2, 1.5]

X_list, y_list = [], []
for cls_idx, n in enumerate(class_counts):
    samples = np.random.randn(n, 5) * feature_stds + feature_means[cls_idx]
    X_list.append(samples)
    y_list.extend([cls_idx] * n)

X = np.vstack(X_list)
y = np.array(y_list)

# Clip to realistic ranges
X[:, 0] = np.clip(X[:, 0], 1, 1000)    # word count >= 1
X[:, 1] = np.clip(X[:, 1], 0, 50)      # link count >= 0
X[:, 2] = np.clip(X[:, 2], 0, 1)       # binary-ish probability
X[:, 3] = np.clip(X[:, 3], 0, 20)      # exclamation marks >= 0
X[:, 4] = np.clip(X[:, 4], 0, 10)      # reply count >= 0

# Shuffle
shuffle_idx = np.random.permutation(n_samples)
X, y = X[shuffle_idx], y[shuffle_idx]

feature_names = ['word_count', 'link_count', 'sender_known', 'exclamation', 'reply_count']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"\nTrain: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# Visualise the class distribution and feature separation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of class counts
counts_in_data = np.bincount(y)
axes[0].bar(CLASS_NAMES, counts_in_data, color=CLASS_COLORS, alpha=0.85, edgecolor='white')
for i, cnt in enumerate(counts_in_data):
    axes[0].text(i, cnt + 10, f'{cnt}\n({cnt/n_samples*100:.0f}%)',
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Email Class Distribution\n(Intentionally Imbalanced)', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, max(counts_in_data) * 1.18)

# Box plot of word_count by class
bp_data = [X[y == cls, 0] for cls in range(N_CLASSES)]
bp = axes[1].boxplot(bp_data, labels=CLASS_NAMES, patch_artist=True)
for patch, color in zip(bp['boxes'], CLASS_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Word Count by Email Class\n(Key Discriminating Feature)', fontweight='bold')
axes[1].set_ylabel('Word Count')
axes[1].set_xlabel('Email Class')

plt.suptitle('Email Dataset Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 2: Train the Model and Per-Class Report

### Per-Class Metrics

For each class $k$, treat it as a binary problem: "is this email class $k$ vs everything else?" Then compute:

$$\text{Precision}_k = \frac{TP_k}{TP_k + FP_k} \qquad \text{Recall}_k = \frac{TP_k}{TP_k + FN_k} \qquad F1_k = \frac{2 \cdot P_k \cdot R_k}{P_k + R_k}$$

This gives us one precision, one recall, and one F1 **per class**. The challenge: how do we collapse four F1 scores into one summary number? That's where averaging strategies come in.

In [ ]:
# ---------------------------------------------------------------
# Train Multinomial Logistic Regression
# ---------------------------------------------------------------
clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42))
])
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Overall accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print()

# Full per-class classification report
print("=== Per-Class Classification Report ===")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

## Section 3: Averaging Strategies — From Scratch

### 3.1 Macro Averaging

$$\text{Macro-F1} = \frac{1}{K} \sum_{k=1}^{K} F1_k$$

Each class gets **equal weight** regardless of how many samples it has.

**When to use:** When all classes are equally important to you — you don't want a poor-performing rare class to be hidden by excellent performance on the majority class.

**Problem:** A class with only 10% of the data votes equally with a class having 40% of the data. A single struggling rare class can tank your macro F1 even if you classify 90% of emails correctly.

### 3.2 Micro Averaging

Pool all class-level TP, FP, FN counts into one global count, then compute F1 once:

$$\text{Micro-F1} = \frac{2 \cdot \sum_k TP_k}{2 \cdot \sum_k TP_k + \sum_k FP_k + \sum_k FN_k}$$

Each **sample** gets equal weight — so majority class dominates. For balanced datasets, micro-F1 equals accuracy.

### 3.3 Weighted Averaging

$$\text{Weighted-F1} = \sum_{k=1}^{K} \frac{n_k}{n} \cdot F1_k$$

Each class's F1 is weighted by its **sample count**. Accounts for natural class imbalance — the most common email type (Spam) contributes most to the score.

In [ ]:
# ---------------------------------------------------------------
# Compute per-class P, R, F1 from scratch using confusion matrix
# ---------------------------------------------------------------
def per_class_metrics_scratch(y_true, y_pred, n_classes):
    """Returns per-class precision, recall, F1, and support."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    metrics = []
    for k in range(n_classes):
        tp = cm[k, k]                      # correct predictions for class k
        fp = cm[:, k].sum() - tp           # other classes predicted as k
        fn = cm[k, :].sum() - tp           # class k predicted as something else
        support = cm[k, :].sum()           # total true instances of class k

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)
        metrics.append({'precision': precision, 'recall': recall,
                        'f1': f1, 'support': support,
                        'tp': tp, 'fp': fp, 'fn': fn})
    return metrics

per_class = per_class_metrics_scratch(y_test, y_pred, N_CLASSES)

print("=== Per-Class Metrics (Computed from Scratch) ===")
print(f"{'Class':15s} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("-" * 57)
for cls_idx, m in enumerate(per_class):
    print(f"{CLASS_NAMES[cls_idx]:15s} {m['precision']:10.4f} {m['recall']:10.4f} "
          f"{m['f1']:10.4f} {m['support']:10d}")

In [ ]:
# ---------------------------------------------------------------
# Compute macro, micro, weighted F1 from scratch
# ---------------------------------------------------------------

def macro_f1_scratch(per_class_metrics):
    """Unweighted average of per-class F1 scores."""
    return np.mean([m['f1'] for m in per_class_metrics])

def micro_f1_scratch(per_class_metrics):
    """Pool global TP, FP, FN then compute F1 once."""
    total_tp = sum(m['tp'] for m in per_class_metrics)
    total_fp = sum(m['fp'] for m in per_class_metrics)
    total_fn = sum(m['fn'] for m in per_class_metrics)
    micro_p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    return (2 * micro_p * micro_r / (micro_p + micro_r)
            if (micro_p + micro_r) > 0 else 0.0)

def weighted_f1_scratch(per_class_metrics):
    """Class-frequency-weighted average of per-class F1 scores."""
    total_support = sum(m['support'] for m in per_class_metrics)
    return sum(m['f1'] * m['support'] / total_support for m in per_class_metrics)

macro_f1   = macro_f1_scratch(per_class)
micro_f1   = micro_f1_scratch(per_class)
weighted_f1 = weighted_f1_scratch(per_class)

# Verify vs sklearn
sk_macro   = f1_score(y_test, y_pred, average='macro')
sk_micro   = f1_score(y_test, y_pred, average='micro')
sk_weighted = f1_score(y_test, y_pred, average='weighted')

print("=== Averaging Strategy Comparison ===")
print(f"{'Metric':20s} {'Scratch':>10} {'sklearn':>10} {'Match?':>8}")
print("-" * 52)
for name, scratch, sk in [
    ('Macro F1',    macro_f1,    sk_macro),
    ('Micro F1',    micro_f1,    sk_micro),
    ('Weighted F1', weighted_f1, sk_weighted),
]:
    match = "YES" if abs(scratch - sk) < 1e-6 else "NO"
    print(f"{name:20s} {scratch:10.6f} {sk:10.6f} {match:>8}")

print()
print("Interpretation:")
print(f"  Macro F1 = {macro_f1:.4f}  — equal weight to all 4 classes")
print(f"  Micro F1 = {micro_f1:.4f}  — driven by majority class (Spam 40%)")
print(f"  Weighted = {weighted_f1:.4f}  — accounts for class frequencies")
print(f"  Accuracy = {acc:.4f}  — for balanced data, ≈ micro F1")

## Section 4: Multi-Class Confusion Matrix

In binary classification, the confusion matrix is 2×2. For K classes, it becomes a **K×K matrix**:

- **Rows** = Actual class
- **Columns** = Predicted class
- **Diagonal** = Correct predictions
- **Off-diagonal** = Errors (row class was misclassified as column class)

**Normalised version** (% per row): for each actual class, what fraction was predicted as each class? This reveals systematic confusions — e.g., "Personal emails are often predicted as Work."

In [ ]:
# ---------------------------------------------------------------
# 4x4 Confusion Matrix: raw counts and row-normalised
# ---------------------------------------------------------------
cm_raw  = confusion_matrix(y_test, y_pred)
cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)   # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw count heatmap
sns.heatmap(
    cm_raw, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, linecolor='gray', ax=axes[0], cbar=True
)
axes[0].set_title('Confusion Matrix — Raw Counts', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Predicted Class', fontsize=11)
axes[0].set_ylabel('Actual Class', fontsize=11)

# Normalised heatmap
sns.heatmap(
    cm_norm, annot=True, fmt='.2%', cmap='Greens',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, linecolor='gray', ax=axes[1], cbar=True,
    vmin=0, vmax=1
)
axes[1].set_title('Confusion Matrix — Row Normalised (% per Actual Class)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Predicted Class', fontsize=11)
axes[1].set_ylabel('Actual Class', fontsize=11)

plt.suptitle('Email Classifier: Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Reading tip: Each row shows what % of that actual class was predicted as each predicted class.")
print("Diagonal entries = recall for that class. Off-diagonal = systematic mistakes.")

## Section 5: When Macro, Micro, and Weighted Disagree

The three averaging strategies tell **different stories**. The key insight:

> A model can have **macro F1 = 0.40** but **weighted F1 = 0.88** — because it's excellent on the 40% majority class and terrible on the 10% minority class.

We'll construct three synthetic models to illustrate this divergence:
1. **Model A:** Strong on majority class, terrible on minority (common problem with imbalanced data)
2. **Model B:** Balanced — mediocre on all classes equally
3. **Model C:** Excellent overall but one class it completely fails

In [ ]:
# ---------------------------------------------------------------
# Simulate three model prediction patterns to show disagreement
# ---------------------------------------------------------------
rng = np.random.default_rng(seed=7)
n_test = len(y_test)
class_support = np.bincount(y_test)   # [n_spam, n_promo, n_personal, n_work]

def synthetic_predictions(y_true, per_class_accuracy, rng_obj):
    """Generate synthetic y_pred where class k has given accuracy."""
    y_pred_syn = y_true.copy()
    for cls, acc in enumerate(per_class_accuracy):
        mask = (y_true == cls)
        n_wrong = int(mask.sum() * (1 - acc))
        wrong_indices = rng_obj.choice(np.where(mask)[0], size=n_wrong, replace=False)
        # Assign wrong predictions randomly from other classes
        other_classes = [c for c in range(N_CLASSES) if c != cls]
        y_pred_syn[wrong_indices] = rng_obj.choice(other_classes, size=n_wrong)
    return y_pred_syn

# Model A: great on Spam (majority), terrible on Work (minority)
y_pred_A = synthetic_predictions(y_test, [0.95, 0.85, 0.50, 0.10], rng)

# Model B: consistently mediocre across all classes
y_pred_B = synthetic_predictions(y_test, [0.65, 0.65, 0.65, 0.65], rng)

# Model C: excellent everywhere but completely fails Work class
y_pred_C = synthetic_predictions(y_test, [0.92, 0.90, 0.88, 0.02], rng)

model_preds = {'Model A (majority-biased)': y_pred_A,
               'Model B (balanced mediocre)': y_pred_B,
               'Model C (one class failure)': y_pred_C,
               'Our LR Model':                y_pred}

disagreement_rows = []
for mname, yp in model_preds.items():
    disagreement_rows.append({
        'Model': mname,
        'Accuracy': round(accuracy_score(y_test, yp), 4),
        'Macro F1':    round(f1_score(y_test, yp, average='macro'),    4),
        'Micro F1':    round(f1_score(y_test, yp, average='micro'),     4),
        'Weighted F1': round(f1_score(y_test, yp, average='weighted'),  4),
    })

dis_df = pd.DataFrame(disagreement_rows).set_index('Model')
print("=== When Macro, Micro, and Weighted F1 Tell Different Stories ===")
print(dis_df.to_string())
print()
print("Key observations:")
print("  Model A: weighted F1 >> macro F1 — great on majority, terrible on minority")
print("  Model B: all three metrics agree — class balance in error rates")
print("  Model C: macro F1 collapses when one class is ignored entirely")

In [ ]:
# ---------------------------------------------------------------
# Visualise the divergence between averaging strategies
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))

metrics_to_plot = ['Macro F1', 'Micro F1', 'Weighted F1']
model_names     = list(dis_df.index)
x = np.arange(len(model_names))
width = 0.22
colors = ['steelblue', 'tomato', 'mediumseagreen']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, dis_df[metric].values, width,
                  label=metric, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.1)
ax.set_title('Macro vs Micro vs Weighted F1: Same Data, Different Stories',
             fontweight='bold', fontsize=12)
ax.legend(fontsize=11)
ax.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()
print("Neither metric is universally correct — they answer different business questions.")

## Section 6: Per-Class F1 Bar Chart — Where Does the Model Struggle?

A bar chart sorted by class size reveals whether the model's problems are concentrated in **rare classes** (common) or spread evenly.

In [ ]:
# ---------------------------------------------------------------
# Per-class F1 bar chart, sorted by class frequency (largest first)
# ---------------------------------------------------------------
per_class_f1       = [m['f1']      for m in per_class]
per_class_support  = [m['support'] for m in per_class]
per_class_precision = [m['precision'] for m in per_class]
per_class_recall   = [m['recall']  for m in per_class]

# Sort by support descending
sort_order = np.argsort(per_class_support)[::-1]
sorted_names     = [CLASS_NAMES[i]       for i in sort_order]
sorted_f1        = [per_class_f1[i]      for i in sort_order]
sorted_precision = [per_class_precision[i] for i in sort_order]
sorted_recall    = [per_class_recall[i]  for i in sort_order]
sorted_support   = [per_class_support[i] for i in sort_order]
sorted_colors    = [CLASS_COLORS[i]      for i in sort_order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# F1 by class, sorted by frequency
bars = axes[0].bar(sorted_names, sorted_f1, color=sorted_colors, alpha=0.85, edgecolor='white')
for bar, f1_val, sup in zip(bars, sorted_f1, sorted_support):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'F1={f1_val:.2f}\n(n={sup})', ha='center', fontsize=9)
axes[0].set_ylim(0, 1.18)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Per-Class F1 Score\n(Sorted by Class Frequency)', fontweight='bold')
axes[0].axhline(macro_f1,    color='steelblue', linestyle='--', label=f'Macro F1={macro_f1:.2f}')
axes[0].axhline(weighted_f1, color='tomato',    linestyle='--', label=f'Weighted F1={weighted_f1:.2f}')
axes[0].legend(fontsize=9)

# Precision vs Recall by class
x_idx = np.arange(N_CLASSES)
width = 0.3
axes[1].bar(x_idx - width/2, sorted_precision, width, label='Precision',
            color=sorted_colors, alpha=0.85, edgecolor='white')
axes[1].bar(x_idx + width/2, sorted_recall,    width, label='Recall',
            color=sorted_colors, alpha=0.45, edgecolor='gray')
axes[1].set_xticks(x_idx)
axes[1].set_xticklabels(sorted_names)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel('Score')
axes[1].set_title('Precision vs Recall per Class\n(Solid=Precision, Faded=Recall)', fontweight='bold')

plt.suptitle('Where Does the Email Classifier Struggle?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7: Cohen's Kappa — Agreement Beyond Chance

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where:
- $p_o$ = observed accuracy (proportion of correctly classified samples)
- $p_e$ = expected accuracy if both the model and the labels were assigned **randomly** based on the observed class distributions

| Kappa | Interpretation |
|-------|----------------|
| < 0   | Worse than random chance |
| 0.0   | No better than random chance |
| 0.2–0.4 | Fair agreement |
| 0.4–0.6 | Moderate agreement |
| 0.6–0.8 | Substantial agreement |
| 0.8–1.0 | Near-perfect agreement |
| 1.0   | Perfect classification |

**Why it matters for imbalanced data:** If Spam is 40% of emails, a model that predicts *everything* as Spam gets 40% accuracy. Kappa accounts for this, giving it κ ≈ 0 despite high "accuracy."

In [ ]:
# ---------------------------------------------------------------
# Cohen's Kappa — implement from scratch and verify vs sklearn
# ---------------------------------------------------------------
def cohen_kappa_scratch(y_true, y_pred):
    """Compute Cohen's Kappa from scratch.
    p_o = observed agreement (accuracy)
    p_e = expected agreement by chance based on marginal distributions
    """
    n = len(y_true)
    classes = np.unique(np.concatenate([y_true, y_pred]))
    K = len(classes)

    # Build confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Observed accuracy: sum of diagonal / total
    p_o = np.trace(cm) / n

    # Expected accuracy: for each class k, P(true=k) * P(pred=k)
    row_sums = cm.sum(axis=1)   # true class counts
    col_sums = cm.sum(axis=0)   # predicted class counts
    p_e = np.sum((row_sums / n) * (col_sums / n))

    kappa = (p_o - p_e) / (1 - p_e) if (1 - p_e) != 0 else 1.0
    return kappa, p_o, p_e

kappa_scratch, p_o, p_e = cohen_kappa_scratch(y_test, y_pred)
kappa_sklearn  = cohen_kappa_score(y_test, y_pred)

print("=== Cohen's Kappa ===")
print(f"  Observed accuracy (p_o): {p_o:.4f}")
print(f"  Expected by chance (p_e): {p_e:.4f}")
print(f"  Cohen's Kappa (scratch):  {kappa_scratch:.4f}")
print(f"  Cohen's Kappa (sklearn):  {kappa_sklearn:.4f}")
print(f"  Match: {abs(kappa_scratch - kappa_sklearn) < 1e-6}")
print()

# Compute kappa for all 4 model scenarios
print("=== Kappa for All Model Scenarios ===")
print(f"{'Model':30s} {'Accuracy':>10} {'Kappa':>10}")
print("-" * 53)
for mname, yp in model_preds.items():
    k, _, _ = cohen_kappa_scratch(y_test, yp)
    a = accuracy_score(y_test, yp)
    print(f"{mname:30s} {a:10.4f} {k:10.4f}")

In [ ]:
# ---------------------------------------------------------------
# Demonstrate: high accuracy ≠ high kappa (imbalanced class problem)
# A classifier that always predicts "Spam" (majority class)
# ---------------------------------------------------------------
y_pred_allspam = np.zeros_like(y_test)   # always predict class 0 = Spam
acc_allspam    = accuracy_score(y_test, y_pred_allspam)
kappa_allspam, _, _ = cohen_kappa_scratch(y_test, y_pred_allspam)

print("=== Degenerate Model: Always Predicts 'Spam' ===")
print(f"  Accuracy: {acc_allspam:.4f} ({acc_allspam*100:.1f}%)  <-- looks decent!")
print(f"  Kappa:    {kappa_allspam:.4f}               <-- reveals it's no better than chance")
print(f"  Macro F1: {f1_score(y_test, y_pred_allspam, average='macro'):.4f}  <-- macro also catches this")
print()
print("Lesson: When classes are imbalanced, accuracy alone is misleading.")
print("Kappa = 0.0 means: this model does no better than a random guesser")
print("that knows the class distribution.")

# Visual: kappa vs accuracy scatter for our model scenarios
scenarios = list(model_preds.items()) + [('Always-Spam Baseline', y_pred_allspam)]
acc_vals   = [accuracy_score(y_test, yp)            for _, yp in scenarios]
kappa_vals = [cohen_kappa_scratch(y_test, yp)[0]    for _, yp in scenarios]
names      = [n for n, _ in scenarios]

fig, ax = plt.subplots(figsize=(9, 5))
sc_colors = ['steelblue', 'tomato', 'mediumseagreen', 'darkorchid', 'gray']
for i, (name, acc_v, kap_v) in enumerate(zip(names, acc_vals, kappa_vals)):
    ax.scatter(acc_v, kap_v, s=140, color=sc_colors[i], zorder=5, edgecolor='white', linewidth=1.2)
    ax.annotate(name, (acc_v, kap_v), textcoords='offset points',
                xytext=(8, 4), fontsize=9)

ax.axhline(0, color='red', linestyle='--', alpha=0.6, label='Kappa=0 (no better than chance)')
ax.set_xlabel('Accuracy', fontsize=12)
ax.set_ylabel("Cohen's Kappa", fontsize=12)
ax.set_title("Accuracy vs Cohen's Kappa\n(High accuracy ≠ high kappa with imbalanced classes)",
             fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Section 8: Full Metric Summary and Choosing the Right Average

### Decision Guide: Which Averaging Strategy to Use?

| Scenario | Use |
|----------|-----|
| All classes equally important, regardless of frequency | **Macro F1** |
| Overall correct classification rate matters; majority class is most important | **Micro F1** |
| Account for natural class distribution; most common in practice | **Weighted F1** |
| Imbalanced data; need to measure agreement beyond chance | **Cohen's Kappa** |

### Our Email Classifier Summary

In [ ]:
# ---------------------------------------------------------------
# Final comprehensive summary of our email classifier
# ---------------------------------------------------------------
print("=" * 60)
print("EMAIL CLASSIFIER — FULL EVALUATION SUMMARY")
print("=" * 60)
print()
print("Dataset:")
print(f"  2,000 emails | 4 classes | 40/30/20/10 split")
print(f"  Test set: {len(y_test)} samples (25% stratified)")
print()
print("Per-Class Performance:")
header = f"  {'Class':15s} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}"
print(header)
print("  " + "-" * 55)
for cls_idx, m in enumerate(per_class):
    print(f"  {CLASS_NAMES[cls_idx]:15s} {m['precision']:10.4f} {m['recall']:10.4f} "
          f"{m['f1']:10.4f} {m['support']:10d}")
print()
print("Summary Metrics:")
print(f"  Accuracy:     {acc:.4f}")
print(f"  Macro F1:     {macro_f1:.4f}  (equal weight per class)")
print(f"  Micro F1:     {micro_f1:.4f}  (equal weight per sample)")
print(f"  Weighted F1:  {weighted_f1:.4f}  (weighted by class frequency)")
print(f"  Cohen's Kappa: {kappa_sklearn:.4f}  (agreement beyond chance)")
print()
print("Business Recommendation:")
print("  - Report Weighted F1 on your dashboard (accounts for class distribution)")
print("  - Monitor Macro F1 to ensure minority classes aren't neglected")
print("  - Use Kappa in academic/comparative reports for rigor")
print("  - Always show the per-class report to the engineering team")

In [ ]:
# ---------------------------------------------------------------
# Final visual: comprehensive 4-panel dashboard
# ---------------------------------------------------------------
fig = plt.figure(figsize=(15, 10))

# Panel 1: Per-class F1 bar
ax1 = fig.add_subplot(2, 2, 1)
ax1.bar(CLASS_NAMES, per_class_f1, color=CLASS_COLORS, alpha=0.85, edgecolor='white')
ax1.axhline(macro_f1,    color='navy',  linestyle='--', label=f'Macro F1={macro_f1:.3f}')
ax1.axhline(weighted_f1, color='green', linestyle='--', label=f'Weighted F1={weighted_f1:.3f}')
ax1.set_ylim(0, 1.15)
ax1.set_title('Per-Class F1 Score', fontweight='bold')
ax1.set_ylabel('F1')
ax1.legend(fontsize=9)
for i, f in enumerate(per_class_f1):
    ax1.text(i, f + 0.02, f'{f:.3f}', ha='center', fontsize=10)

# Panel 2: Normalised confusion matrix
ax2 = fig.add_subplot(2, 2, 2)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=ax2, cbar=False)
ax2.set_title('Confusion Matrix (Row-Normalised)', fontweight='bold')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')

# Panel 3: Macro vs Micro vs Weighted across scenarios
ax3 = fig.add_subplot(2, 2, 3)
x3 = np.arange(len(dis_df))
w3 = 0.22
ax3.bar(x3 - w3, dis_df['Macro F1'],    w3, label='Macro',    color='steelblue',      alpha=0.85)
ax3.bar(x3,      dis_df['Micro F1'],    w3, label='Micro',    color='tomato',         alpha=0.85)
ax3.bar(x3 + w3, dis_df['Weighted F1'], w3, label='Weighted', color='mediumseagreen', alpha=0.85)
ax3.set_xticks(x3)
ax3.set_xticklabels(dis_df.index, rotation=20, ha='right', fontsize=9)
ax3.set_ylim(0, 1.1)
ax3.set_title('Macro / Micro / Weighted F1 by Model Scenario', fontweight='bold')
ax3.legend(fontsize=9)

# Panel 4: Kappa vs Accuracy
ax4 = fig.add_subplot(2, 2, 4)
for i, (name, acc_v, kap_v) in enumerate(zip(names, acc_vals, kappa_vals)):
    ax4.scatter(acc_v, kap_v, s=130, color=sc_colors[i], zorder=5, edgecolor='white', linewidth=1.2)
    ax4.annotate(name, (acc_v, kap_v), textcoords='offset points',
                 xytext=(6, 4), fontsize=8.5)
ax4.axhline(0, color='red', linestyle='--', alpha=0.6)
ax4.set_xlabel('Accuracy')
ax4.set_ylabel("Cohen's Kappa")
ax4.set_title("Accuracy vs Cohen's Kappa", fontweight='bold')

plt.suptitle('Email Classifier — Complete Evaluation Dashboard',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Self-Check Questions

Work through these before moving on.

---

**Q1.** You have a 4-class email classifier. Class A has 1,000 samples and F1 = 0.95. Class B has only 50 samples and F1 = 0.20. Macro F1 = 0.575; Weighted F1 = 0.93. Which metric should a business dashboard report? Why?

<details>
<summary>Answer</summary>

This depends on the business context — and that's the honest answer.

**If the business cares proportionally more about Class A** (because it dominates email volume), use **Weighted F1 = 0.93**. It reflects real-world performance weighted by email frequency — a \$93-in-every-100 success rate on emails actually seen.

**However**, reporting *only* weighted F1 = 0.93 on the dashboard **hides the Class B problem**. Class B (F1 = 0.20) is being almost completely misclassified. If Class B contains critical emails — say, security alerts or executive communications — missing 80% of them is a severe failure even if they're rare.

**Best practice for a business dashboard:** Report Weighted F1 as the headline metric *plus* a per-class breakdown in a table below. Flag any class where F1 < 0.50 with a visual warning. This way, business users see the summary but aren't misled about edge cases.

</details>

---

**Q2.** Your model has Micro F1 = 0.82 but Macro F1 = 0.41. What does this tell you about class-level performance?

<details>
<summary>Answer</summary>

This is a classic sign of **severe class imbalance combined with biased predictions**.

- **Micro F1 = 0.82** means the model correctly classifies 82% of emails *by count* — but because Micro is dominated by the majority class, this largely reflects good performance on the most frequent email type.
- **Macro F1 = 0.41** means that when you average F1 across all classes **equally**, the result collapses. This happens when one or more **minority classes** have very low F1 scores that drag the unweighted average down.

In concrete terms: if you have classes with F1 scores of [0.90, 0.85, 0.80, 0.05], the macro average is (0.90+0.85+0.80+0.05)/4 = 0.65 — but if the 0.05 class is very rare, micro F1 would still be high.

**Action:** Investigate which class(es) have F1 near zero. Common causes: the model never predicts that class at all (under-sampling), or the class features heavily overlap with another class. Solutions: oversample minority class, adjust class weights in the loss function, or collect more data for the rare class.

</details>

---

**Q3.** Your email classifier has Accuracy = 0.80 but Cohen's Kappa = 0.12. What does the low Kappa reveal that accuracy is hiding?

<details>
<summary>Answer</summary>

A Cohen's Kappa of 0.12 ("slight agreement") despite 80% accuracy reveals that the model's correct predictions are **largely explainable by chance alone**, given the class distribution.

Here's what's happening: if Spam makes up 75–80% of your email dataset, a model that **always predicts Spam** achieves 75–80% accuracy automatically — without learning anything about email content. Kappa = 0.12 says: after accounting for what a random guesser would get (by knowing that 80% of emails are Spam), the model is only barely better than random.

Concretely, Kappa = (p_o - p_e) / (1 - p_e). If p_e (expected accuracy by chance) is 0.78 and p_o (observed accuracy) is 0.80, then κ = (0.80 - 0.78)/(1 - 0.78) ≈ 0.09 — nearly zero. The model has learned almost nothing beyond the base rate.

**What to do:** Check per-class confusion matrix — likely the model almost never predicts minority classes. Strategies: balance the dataset (SMOTE, undersampling), add class weights to the loss function, or check if the features actually discriminate between classes. Always report Kappa alongside accuracy for imbalanced multi-class problems.

</details>